# Food Delivery Time Prediction

This project predicts whether a food order will be **Fast** or **Delayed**.

The dataset contains information about distance, traffic, weather, ratings,
order details and delivery time.

Since the dataset does not have a Fast/Delayed column, the median delivery
time is used to create the two classes.

In [ ]:
import ast
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, roc_auc_score
)

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

np.random.seed(42)
random.seed(42)
tf.random.set_seed(42)

df = pd.read_csv("Food_Delivery_Time_Prediction(2).csv")

print("Shape:", df.shape)
df.head()

## Looking at the data

In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nData types:")
df.dtypes

In [ ]:
df.describe()

In [ ]:
# Check the main categorical columns
for col in ["Weather_Conditions", "Traffic_Conditions",
            "Order_Priority", "Order_Time", "Vehicle_Type"]:
    print("\n", col)
    print(df[col].value_counts())

## Creating the target column

There is no Fast/Delayed label in the original file.

I used the median delivery time as the cutoff:
- delivery time <= median: Fast
- delivery time > median: Delayed

In [ ]:
median_time = df["Delivery_Time"].median()

df["Delivery_Status"] = np.where(
    df["Delivery_Time"] <= median_time,
    "Fast",
    "Delayed"
)

df["Target"] = (df["Delivery_Status"] == "Delayed").astype(int)

print("Median delivery time:", median_time)
print(df["Delivery_Status"].value_counts())

## A few extra features

The dataset already has distance, so there is no need to calculate it again.

The location columns are used later for the CNN image representation.

In [ ]:
def get_coordinates(value):
    try:
        return ast.literal_eval(value)
    except:
        return (np.nan, np.nan)

customer = df["Customer_Location"].apply(get_coordinates)
restaurant = df["Restaurant_Location"].apply(get_coordinates)

df["Customer_Lat"] = customer.apply(lambda x: x[0])
df["Customer_Lon"] = customer.apply(lambda x: x[1])

df["Restaurant_Lat"] = restaurant.apply(lambda x: x[0])
df["Restaurant_Lon"] = restaurant.apply(lambda x: x[1])

# Simple time and traffic features
time_values = {
    "Morning": 8,
    "Afternoon": 13,
    "Evening": 18,
    "Night": 22
}

df["Order_Hour"] = df["Order_Time"].map(time_values)
df["Rush_Hour"] = df["Order_Time"].isin(["Morning", "Evening"]).astype(int)

df[[
    "Distance", "Order_Time", "Order_Hour", "Rush_Hour",
    "Traffic_Conditions", "Weather_Conditions"
]].head()

## Some plots

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="Delivery_Status")
plt.title("Fast and Delayed Orders")
plt.xlabel("Delivery Status")
plt.ylabel("Number of Orders")
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(
    data=df,
    x="Distance",
    y="Delivery_Time",
    hue="Traffic_Conditions"
)
plt.title("Distance vs Delivery Time")
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
sns.boxplot(data=df, x="Traffic_Conditions", y="Delivery_Time")
plt.title("Delivery Time for Different Traffic Conditions")
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
sns.boxplot(data=df, x="Weather_Conditions", y="Delivery_Time")
plt.title("Delivery Time for Different Weather Conditions")
plt.show()

In [ ]:
numeric_cols = [
    "Distance",
    "Delivery_Person_Experience",
    "Restaurant_Rating",
    "Customer_Rating",
    "Order_Cost",
    "Tip_Amount",
    "Traffic_Level",
    "Weather_Level",
    "Rush_Hour"
]

# Add simple numeric versions for the heatmap
traffic_map = {"Low": 0, "Medium": 1, "High": 2}
weather_map = {"Sunny": 0, "Cloudy": 1, "Rainy": 2, "Snowy": 3}

df["Traffic_Level"] = df["Traffic_Conditions"].map(traffic_map)
df["Weather_Level"] = df["Weather_Conditions"].map(weather_map)

plt.figure(figsize=(9, 6))
sns.heatmap(df[numeric_cols + ["Delivery_Time"]].corr(),
            annot=True, fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

## Logistic Regression

First I used Logistic Regression as a simple baseline model.

`Delivery_Time` is not included because it was used to create the target.

In [ ]:
features = [
    "Distance",
    "Delivery_Person_Experience",
    "Restaurant_Rating",
    "Customer_Rating",
    "Order_Cost",
    "Tip_Amount",
    "Order_Time",
    "Weather_Conditions",
    "Traffic_Conditions",
    "Order_Priority",
    "Vehicle_Type",
    "Order_Hour",
    "Rush_Hour"
]

X = df[features]
y = df["Target"]

categorical = [
    "Order_Time",
    "Weather_Conditions",
    "Traffic_Conditions",
    "Order_Priority",
    "Vehicle_Type"
]

numerical = [
    "Distance",
    "Delivery_Person_Experience",
    "Restaurant_Rating",
    "Customer_Rating",
    "Order_Cost",
    "Tip_Amount",
    "Order_Hour",
    "Rush_Hour"
]

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numerical),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical)
])

logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

logistic_model.fit(X_train, y_train)

y_pred = logistic_model.predict(X_test)
y_prob = logistic_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))

In [ ]:
logistic_results = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1 Score": f1_score(y_test, y_pred),
    "ROC-AUC": roc_auc_score(y_test, y_prob)
}

pd.Series(logistic_results)

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Logistic Regression Confusion Matrix")
plt.show()

In [ ]:
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob)

plt.figure(figsize=(6, 5))
plt.plot(fpr_lr, tpr_lr, label=f"AUC = {roc_auc_score(y_test, y_prob):.2f}")
plt.plot([0, 1], [0, 1], "--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Logistic Regression ROC Curve")
plt.legend()
plt.show()

## 5-fold cross-validation

The dataset is small, so I also checked the Logistic Regression model with
5-fold stratified cross-validation.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(
    logistic_model,
    X,
    y,
    cv=cv,
    scoring="accuracy"
)

print("Fold accuracies:", np.round(cv_scores, 3))
print("Mean accuracy:", round(cv_scores.mean(), 3))

## CNN

For the CNN, the two locations are converted into a small image.

The customer location is shown as one point, the restaurant location as
another point, and the route between them is drawn as a line.

This gives the CNN an image representation of the delivery route.

In [ ]:
IMG_SIZE = 64

def make_route_image(row):
    img = np.zeros((IMG_SIZE, IMG_SIZE, 1), dtype=np.float32)

    c_lat = row["Customer_Lat"]
    c_lon = row["Customer_Lon"]
    r_lat = row["Restaurant_Lat"]
    r_lon = row["Restaurant_Lon"]

    points_lat = np.array([c_lat, r_lat])
    points_lon = np.array([c_lon, r_lon])

    lat_min, lat_max = points_lat.min(), points_lat.max()
    lon_min, lon_max = points_lon.min(), points_lon.max()

    lat_range = lat_max - lat_min
    lon_range = lon_max - lon_min

    if lat_range == 0:
        lat_range = 1e-8
    if lon_range == 0:
        lon_range = 1e-8

    def convert(lat, lon):
        x = int((lon - lon_min) / lon_range * (IMG_SIZE - 1))
        y = int((lat - lat_min) / lat_range * (IMG_SIZE - 1))
        return x, IMG_SIZE - 1 - y

    x1, y1 = convert(c_lat, c_lon)
    x2, y2 = convert(r_lat, r_lon)

    # draw the route
    n = max(abs(x2 - x1), abs(y2 - y1)) + 1
    xs = np.linspace(x1, x2, n).astype(int)
    ys = np.linspace(y1, y2, n).astype(int)

    img[ys, xs, 0] = 0.5

    # customer
    img[max(0, y1-2):min(IMG_SIZE, y1+3),
        max(0, x1-2):min(IMG_SIZE, x1+3), 0] = 1.0

    # restaurant
    img[max(0, y2-2):min(IMG_SIZE, y2+3),
        max(0, x2-2):min(IMG_SIZE, x2+3), 0] = 0.8

    return img

images = np.array([make_route_image(row) for _, row in df.iterrows()])
labels = df["Target"].values

print("Image data shape:", images.shape)

In [ ]:
# Check one generated route image
plt.figure(figsize=(5, 5))
plt.imshow(images[0].squeeze(), cmap="gray")
plt.axis("off")
plt.title("Example Delivery Route Image")
plt.show()

In [ ]:
X_img_train, X_img_test, y_img_train, y_img_test = train_test_split(
    images,
    labels,
    test_size=0.20,
    random_state=42,
    stratify=labels
)

cnn = Sequential([
    Conv2D(16, (3, 3), activation="relu", input_shape=(IMG_SIZE, IMG_SIZE, 1)),
    MaxPooling2D((2, 2)),

    Conv2D(32, (3, 3), activation="relu"),
    MaxPooling2D((2, 2)),

    Flatten(),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid")
])

cnn.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

cnn.summary()

In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history = cnn.fit(
    X_img_train,
    y_img_train,
    validation_split=0.2,
    epochs=30,
    batch_size=16,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history.history["accuracy"], label="Training")
plt.plot(history.history["val_accuracy"], label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("CNN Training Accuracy")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history.history["loss"], label="Training")
plt.plot(history.history["val_loss"], label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("CNN Training Loss")
plt.legend()
plt.show()

In [ ]:
cnn_prob = cnn.predict(X_img_test, verbose=0).ravel()
cnn_pred = (cnn_prob >= 0.5).astype(int)

cnn_results = {
    "Accuracy": accuracy_score(y_img_test, cnn_pred),
    "Precision": precision_score(y_img_test, cnn_pred),
    "Recall": recall_score(y_img_test, cnn_pred),
    "F1 Score": f1_score(y_img_test, cnn_pred),
    "ROC-AUC": roc_auc_score(y_img_test, cnn_prob)
}

pd.Series(cnn_results)

In [ ]:
cm_cnn = confusion_matrix(y_img_test, cnn_pred)

plt.figure(figsize=(5, 4))
sns.heatmap(cm_cnn, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("CNN Confusion Matrix")
plt.show()

In [ ]:
fpr_cnn, tpr_cnn, _ = roc_curve(y_img_test, cnn_prob)

plt.figure(figsize=(6, 5))
plt.plot(fpr_cnn, tpr_cnn, label=f"AUC = {roc_auc_score(y_img_test, cnn_prob):.2f}")
plt.plot([0, 1], [0, 1], "--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("CNN ROC Curve")
plt.legend()
plt.show()

## Trying a few CNN settings

Because the dataset is small, I kept the tuning simple instead of running
a large search.

In [ ]:
settings = [
    (16, 32, 0.001),
    (32, 64, 0.001),
    (16, 32, 0.0005)
]

tuning_results = []

for filters1, filters2, lr in settings:
    model = Sequential([
        Conv2D(filters1, (3, 3), activation="relu",
               input_shape=(IMG_SIZE, IMG_SIZE, 1)),
        MaxPooling2D((2, 2)),
        Conv2D(filters2, (3, 3), activation="relu"),
        MaxPooling2D((2, 2)),
        Flatten(),
        Dense(64, activation="relu"),
        Dropout(0.3),
        Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    h = model.fit(
        X_img_train, y_img_train,
        validation_split=0.2,
        epochs=15,
        batch_size=16,
        verbose=0,
        callbacks=[early_stop]
    )

    loss, acc = model.evaluate(X_img_test, y_img_test, verbose=0)

    tuning_results.append({
        "Filters": f"{filters1}-{filters2}",
        "Learning Rate": lr,
        "Test Accuracy": acc
    })

pd.DataFrame(tuning_results)

## Model comparison

In [ ]:
comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "CNN"],
    "Accuracy": [
        logistic_results["Accuracy"],
        cnn_results["Accuracy"]
    ],
    "Precision": [
        logistic_results["Precision"],
        cnn_results["Precision"]
    ],
    "Recall": [
        logistic_results["Recall"],
        cnn_results["Recall"]
    ],
    "F1 Score": [
        logistic_results["F1 Score"],
        cnn_results["F1 Score"]
    ],
    "ROC-AUC": [
        logistic_results["ROC-AUC"],
        cnn_results["ROC-AUC"]
    ]
})

comparison.round(3)

In [ ]:
plt.figure(figsize=(8, 5))

x = np.arange(len(comparison))
width = 0.35

plt.bar(x - width/2, comparison["Accuracy"], width, label="Accuracy")
plt.bar(x + width/2, comparison["F1 Score"], width, label="F1 Score")

plt.xticks(x, comparison["Model"])
plt.ylim(0, 1)
plt.ylabel("Score")
plt.title("Model Comparison")
plt.legend()
plt.show()

## Conclusion

The project started with basic data cleaning and feature preparation.
Logistic Regression was used as the baseline, followed by a CNN using
delivery route images.

The final model should be selected based on the test results above rather
than assuming that the CNN will always perform better.

The dataset is relatively small, so the CNN results should be interpreted
carefully. More delivery records and actual route/map images would make the
CNN approach more useful.